# 09 · Body refinement by analysis-by-synthesis

Notebook 08's method on the whole avatar: the joined mesh (Anny's body with the ICT-FaceKit head,
`outputs/fitting/avatar.npz`) is rendered with nvdiffrast, posed by the body fit, compared with
every pixel of the body turntable, and nudged until the renders match:

| What | Shared or per frame | Phase |
| --- | --- | --- |
| Albedo of the body: baked from the frames, baked again every `REBAKE_EVERY` steps | shared | 1 → |
| Light: 2nd-order spherical harmonics, RGB | shared | 1 → |
| Corrections to every fitted bone's rotation, the body's position, the face | per frame | 2 → |
| Smooth shape changes of the body (not the hands, not the neck) | shared | 3 |

First the avatar is joined again with notebook 08's head (`outputs/head_refinement/head.npz`):
its identity, shape changes and own blendshapes, by the same code the fitting stage joins them
with, so the neck seam is cut and bent for this head. The head keeps 08's albedo and wrinkle
maps; without 08's result the fitted head stays and its texture is baked here too.

The body's albedo is baked, not learned texel by texel (the lessons: that turns into noise, and
a 1024-pixel render of the body gives a large texture only scattered gradients). Every frame's
pixels are divided by the light their surface gets and averaged into the texture, weighted by
how squarely the surface faces the camera; texels no frame sees are filled from coarser levels.
The light is then learned against the fixed albedo, mostly from the head, whose albedo 08 made.

The hands are baked like the rest, but left out of the colour comparison: their fingers are
posed once for the whole video and never match every frame. The lessons warn that body shape changes fitted to outlines
hide pose errors; here they are broad, priced and come last, after the poses; set
`ITERATIONS["shape"] = 0` to leave Anny's shape alone.

At the end the refined avatar is written as `outputs/export/refined.avatar` (format 4: no
Gaussians, the learned light, the wrinkle maps), for the viewer: drop it on the page.

In [ ]:
import copy
import dataclasses
import time

import matplotlib.pyplot as plt
import numpy as np
import nvdiffrast.torch as dr
import roma
import torch
import torch.nn.functional as F

from avatar_splats.body_models import ict
from avatar_splats.body_models.anny import AnnyBody
from avatar_splats.body_models.hybrid import (
    ATLAS_TILES,
    BODY_TEXTURE,
    HANDS,
    HEAD_TEXTURE,
    Avatar,
    HeadPlacement,
    build_avatar,
    rest_body,
    subdivide,
)
from avatar_splats.body_models.ict import IctFaceKit
from avatar_splats.body_models.ict import landmark_vertices as ict_landmark_vertices
from avatar_splats.body_models.skinning import linear_blend_skinning
from avatar_splats.config import load_config
from avatar_splats.data.video import read_frames
from avatar_splats.export import (
    avatar_arrays,
    eye_metadata,
    face_hair_metadata,
    grow_face_hair,
    measure_eyes,
    measure_hair_colours,
    write_avatar,
)
from avatar_splats.fitting import HEAD_VIDEOS, BodyFit, HeadFit, fitted_path
from avatar_splats.fitting.body import POSED_BONES
from avatar_splats.fitting.losses import second_difference
from avatar_splats.gaussians import GaussianRecord
from avatar_splats.preprocess import Preprocessed, preprocessed_path
from avatar_splats.training.textures import TextureRecord

CONFIG = "../configs/default.yaml"  # the capture: e.g. "../configs/performer-2.yaml"
config = load_config(CONFIG)
device = "cuda"
out_dir = config.outputs / "body_refinement"
out_dir.mkdir(parents=True, exist_ok=True)

## Settings

Lower `RESOLUTION` or `CHUNK` if the GPU runs out of memory.

In [ ]:
RESOLUTION = 1024  # pixels along the frames' longer side
BODY_TEXTURE_SIZE = 1024  # texels per side of the body's texture: about the frames' detail
HEAD_TILE_SIZE = 512  # texels per side of each of the head atlas's tiles
REBAKE_EVERY = 50  # steps between bakes of the albedo, as the light and poses change
MIN_SHADING = 0.2  # pixels the light reaches less than this are not baked (shadowed)
FILL_WEIGHT = 0.05  # bake weight at which a texel trusts itself over the coarser levels
# The viewer's own eyes (procedural iris, cornea) and mouth (stock colours by region) date from
# the Gaussian pipeline's unusable textures there; False draws the eyes and the inside of the
# mouth from 08's learned texture, lit like the skin, as the notebooks render them.
STOCK_EYES_AND_MOUTH = False
HOLD_OUT = 8  # every HOLD_OUT-th frame is held out
CHUNK = 9  # frames rendered at once
SMOOTHING_STEPS = 60  # neighbour averages applied to the body's shape changes
FROZEN_NEAR = 0.03  # metres: body vertices this near the head or the genitals keep their place
EDGE_BAND = 4  # pixels along the mask's edge left out of the colour comparison
NEAR, FAR = 0.1, 10.0  # metres

ITERATIONS = {"appearance": 300, "pose": 400, "shape": 400}

LEARNING_RATES = {
    "light": 0.01,
    "pose": 0.001,  # radians and metres per step
    "expressions": 0.02,
    "offsets": 0.02,  # millimetres per step, before smoothing
}

WEIGHTS = {
    "photo": 1.0,  # mean absolute colour difference, colours in [0, 1]
    "silhouette": 1.0,  # mean squared difference of the outline and the mask
    "pose_prior": 1.0,  # per radian² of correction summed over bones, and per decimetre²
    "temporal": 1.0,  # second differences of the corrections from frame to frame
    "sparsity": config.fitting.expression_sparsity_weight,
    "offsets": 1e-3,  # per mm² of shape change, averaged over vertices
}

## Data: the avatar, the body fit, the frames, and the refined head

In [ ]:
model = IctFaceKit(config.body_models.ict_facekit).to(device)
body = AnnyBody().to(device)
fitted_avatar = Avatar.load(fitted_path(config, "avatar"))
fit = BodyFit.load(fitted_path(config, "body"))
pre = Preprocessed.load(preprocessed_path(config, "body"))
head_fit = HeadFit.load(fitted_path(config, HEAD_VIDEOS[0]))

width, height = (int(v) for v in fit.image_size)
scale = RESOLUTION / max(width, height)
camera = fit.camera().scaled(scale).to(device)
size = (camera.height, camera.width)

everything = list(read_frames(config.capture.body, step=1))
frames = np.stack([everything[i] for i in pre.frame_indices])
del everything
images = torch.from_numpy(frames).to(device).permute(0, 3, 1, 2).float() / 255
images = F.interpolate(images, size, mode="area").permute(0, 2, 3, 1).contiguous()
masks = F.interpolate(torch.from_numpy(pre.masks).to(device).float()[:, None], size, mode="area")[
    :, 0
]
hard = (masks > 0.5).float()[:, None]
inner = (-F.max_pool2d(-hard, 2 * EDGE_BAND + 1, stride=1, padding=EDGE_BAND))[:, 0]
FRAMES = len(images)

held_out = torch.zeros(FRAMES, dtype=torch.bool)
held_out[HOLD_OUT // 2 :: HOLD_OUT] = True
chunks = [(c, False) for c in torch.nonzero(~held_out)[:, 0].split(CHUNK)]
chunks += [(c, True) for c in torch.nonzero(held_out)[:, 0].split(CHUNK)]

head_path = config.outputs / "head_refinement" / "head.npz"
head = dict(np.load(head_path)) if head_path.exists() else None
print(f"{FRAMES} frames at {camera.width}x{camera.height}, {int(held_out.sum())} held out")
print("refined head:", "yes" if head is not None else "no, the fitted one stays")

## The avatar, joined again with the refined head

The fitting stage placed the head on the body (`HeadPlacement`: a scale, a turn and a shift) and
joined the two at the neck. The placement is read back from the fitted avatar, whose head and
blendshapes are ICT-FaceKit's placed by it; then `build_avatar` joins Anny's body with 08's head,
08's own blendshapes included, exactly as the fitting stage does: the seam is cut and the neck
bent for this head, with no step at the join.

In [ ]:
E_HEAD = model.num_expressions
zeros = torch.zeros(1, E_HEAD, device=device)
fitted_identity = torch.from_numpy(head_fit.identity).float().to(device)
fitted_head = model(fitted_identity[None], zeros)[0]

start, count = int(fitted_avatar.head_start), len(model.neutral)
fitted_basis = torch.from_numpy(fitted_avatar.expression_basis[:E_HEAD, :count]).float().to(device)
placed = torch.linalg.lstsq(
    model.expression_basis.reshape(-1, 3), fitted_basis.reshape(-1, 3)
).solution
head_scale = float(torch.linalg.det(placed) ** (1 / 3))
fitted_rest = (
    torch.from_numpy(fitted_avatar.rest_vertices[start : start + count]).float().to(device)
)
face = slice(0, ict.FACE.stop)  # the face, away from the neck the join bent
shift = (fitted_rest[face] - fitted_head[face] @ placed).median(dim=0).values
placement = HeadPlacement(head_scale, (placed / head_scale).T.contiguous(), shift)
misfit = (fitted_head[face] @ placed + shift - fitted_rest[face]).norm(dim=-1).median()
print(f"placement read back: scale {head_scale:.3f}, face misfit {misfit * 1000:.2f} mm")

head_model = copy.deepcopy(model)
identity = fitted_identity
head_rest = fitted_head
if head is not None:
    identity = torch.from_numpy(head["identity"]).float().to(device)
    head_rest = model(identity[None], zeros)[0] + torch.from_numpy(head["offsets"]).float().to(
        device
    )
    head_model.expression_basis += torch.from_numpy(head["correctives"]).float().to(device)
body_rest = rest_body(body, fit.phenotype, fit.local_changes)
avatar = build_avatar(body, body_rest, head_model, head_rest, placement, config.body_models)
for _ in range(config.body_models.subdivisions):
    avatar = subdivide(avatar, config.body_models.max_bone_influences)

HEAD_START = int(avatar.head_start)
HEAD_COUNT = count
HEAD = slice(HEAD_START, HEAD_START + HEAD_COUNT)
rest = torch.from_numpy(avatar.rest_vertices).float().to(device)
basis = torch.from_numpy(avatar.expression_basis).float().to(device)  # (E, V - HEAD_START, 3)
moved = (
    rest[HEAD] - torch.from_numpy(fitted_avatar.rest_vertices[start : start + count]).to(device)
).norm(dim=-1)
median, largest = moved.median() * 1e3, moved.max() * 1e3
print(f"head moved from the fitted avatar: median {median:.1f} mm, max {largest:.1f} mm")

## The model

Each frame: the head's blendshapes at that frame's weights, then linear blend skinning by the
fit's bone rotations plus that frame's corrections. Albedo from the body's baked texture or the
head's atlas (by triangle), plus the wrinkle maps on the face's tile; colour = albedo × the light
(spherical harmonics) the normal receives, antialiased.

The body's shape changes, like the head's, are free values per vertex averaged with their
neighbours `SMOOTHING_STEPS` times. They are zero on the hands and near the head and the
genitals (both are stitched to the body), and fade in from there.

In [ ]:
faces = torch.from_numpy(avatar.faces).long().to(device)
tri = faces.int().contiguous()
face_textures = torch.from_numpy(avatar.face_textures).long().to(device)
face_parts = torch.from_numpy(avatar.face_parts).long().to(device)
corner_tri = torch.arange(3 * len(tri), device=device, dtype=torch.int32).reshape(-1, 3)
corner_uvs = torch.from_numpy(avatar.uvs).float().to(device).reshape(-1, 2).clone()
corner_uvs[:, 1] = 1 - corner_uvs[:, 1]  # textures' row 0 is their top (v = 1)
corner_uvs = corner_uvs[None].contiguous()
bone_indices = torch.from_numpy(avatar.bone_indices).long().to(device)
bone_weights = torch.from_numpy(avatar.bone_weights).float().to(device)
bind_poses = torch.from_numpy(avatar.bind_poses).float().to(device)
context = dr.RasterizeCudaContext()

# The body's shape changes.
body_faces = faces[(faces < HEAD_START).all(dim=1)]
edges = torch.cat([body_faces[:, [0, 1]], body_faces[:, [1, 2]], body_faces[:, [2, 0]]])
loops = torch.arange(HEAD_START, device=device)[:, None].repeat(1, 2)
edges = torch.unique(torch.cat([edges, edges.flip(1), loops]), dim=0)
degree = torch.bincount(edges[:, 0], minlength=HEAD_START).float()
averaging = torch.sparse_coo_tensor(
    edges.T, 1 / degree[edges[:, 0]], (HEAD_START, HEAD_START), check_invariants=True
).coalesce()


def smooth(values, steps=SMOOTHING_STEPS):
    for _ in range(steps):
        values = torch.sparse.mm(averaging, values)
    return values


with torch.no_grad():
    free = torch.ones(HEAD_START, 1, device=device)
    hand_vertices = torch.unique(faces[face_parts == HANDS])
    free[hand_vertices[hand_vertices < HEAD_START]] = 0
    stitched = rest[HEAD_START:]  # the head and the genitals
    nearest = torch.cat(
        [torch.cdist(part, stitched).min(dim=1).values for part in rest[:HEAD_START].split(4096)]
    )
    free[nearest < FROZEN_NEAR] = 0

MILLIMETRE = 1e-3
offset_values = torch.nn.Parameter(torch.zeros(HEAD_START, 3, device=device))


def body_offsets():
    # (HEAD_START, 3) millimetres
    return smooth(offset_values * free) * free


def rest_vertices():
    return torch.cat([rest[:HEAD_START] + body_offsets() * MILLIMETRE, rest[HEAD_START:]])


# Per frame: corrections of every fitted bone's rotation and of the root's position, and the face.
fit_rotations = torch.from_numpy(fit.rotations).float().to(device)
fit_translations = torch.from_numpy(fit.translation).float().to(device)
posed = torch.zeros(body.num_bones, 1, device=device)
posed[[body.bone(name) for name in POSED_BONES] + [0]] = 1
rotation_corrections = torch.nn.Parameter(torch.zeros_like(fit_rotations))
translation_corrections = torch.nn.Parameter(torch.zeros_like(fit_translations))
expression_logits = torch.nn.Parameter(torch.full((FRAMES, E_HEAD), -5.0, device=device))


def expressions():
    return torch.sigmoid(expression_logits)


def posed_vertices(rest_now, weights, rotations, translations):
    # (B, V, 3) world vertices for (B, E_HEAD) face weights, (B, J, 3) rotations, (B, 3)
    skinning = body.skinning(bind_poses, rotations, translations)
    rest_b = rest_now.expand(len(rotations), -1, -1)
    face = torch.einsum("be,evc->bvc", weights, basis[:E_HEAD, :HEAD_COUNT])
    rest_b = torch.cat([rest_b[:, :HEAD_START], rest_b[:, HEAD] + face, rest_b[:, HEAD.stop :]], 1)
    return linear_blend_skinning(rest_b, bone_indices, bone_weights, skinning)


def frame_pose(frames):
    rotations = fit_rotations[frames] + rotation_corrections[frames] * posed
    return rotations, fit_translations[frames] + translation_corrections[frames]


def vertex_normals(vertices):
    corners = vertices[:, faces]
    face_normals = torch.linalg.cross(
        corners[:, :, 1] - corners[:, :, 0], corners[:, :, 2] - corners[:, :, 0]
    )
    normals = torch.zeros_like(vertices)
    for k in range(3):
        normals = normals.index_add(1, faces[:, k], face_normals)
    return F.normalize(normals, dim=-1)


def sh_basis(n):
    x, y, z = n.unbind(-1)
    return torch.stack(
        [
            0.282095 * torch.ones_like(x),
            0.488603 * y,
            0.488603 * z,
            0.488603 * x,
            1.092548 * x * y,
            1.092548 * y * z,
            0.315392 * (3 * z * z - 1),
            1.092548 * x * z,
            0.546274 * (x * x - y * y),
        ],
        dim=-1,
    )


def uniform_light():
    light = torch.zeros(9, 3, device=device)
    light[0] = 1 / 0.282095
    return light


def to_clip(vertices, camera):
    view = vertices @ camera.view[:3, :3].T + camera.view[:3, 3]
    x, y, z = view.unbind(-1)
    K, W, H = camera.K, camera.width, camera.height
    return torch.stack(
        [
            2 * K[0, 0] / W * x + (2 * K[0, 2] / W - 1) * z,
            2 * K[1, 1] / H * y + (2 * K[1, 2] / H - 1) * z,
            (FAR + NEAR) / (FAR - NEAR) * z - 2 * FAR * NEAR / (FAR - NEAR),
            z,
        ],
        dim=-1,
    ).contiguous()


# Textures: the body's is baked (below); the head's atlas is 08's, its genitals' tile baked.
BODY_SHAPE = (BODY_TEXTURE_SIZE, BODY_TEXTURE_SIZE)
# Avatars built before the genitals have ICT-FaceKit's tiles only; later ones one more.
head_u = avatar.uvs[avatar.face_textures == HEAD_TEXTURE][..., 0].max()
HEAD_TILES = ict.UV_TILES if head_u > ict.UV_TILES / ATLAS_TILES + 1e-3 else ATLAS_TILES
HEAD_SHAPE = (HEAD_TILE_SIZE, HEAD_TILES * HEAD_TILE_SIZE)


def resized(image, shape):
    # (H, W, 3) array -> (1, shape..., 3) tensor
    tensor = torch.as_tensor(image, dtype=torch.float32, device=device).permute(2, 0, 1)[None]
    mode = "area" if tensor.shape[-1] > shape[1] else "bilinear"
    return F.interpolate(tensor, shape, mode=mode).permute(0, 2, 3, 1).contiguous()


with torch.no_grad():
    mean_colour = (images * inner[..., None]).sum((0, 1, 2)) / inner.sum()
    baked = {
        "body": mean_colour.expand(1, *BODY_SHAPE, 3).clone(),
        "head": mean_colour.expand(1, *HEAD_SHAPE, 3).clone(),
    }
    if head is not None:
        ict_tiles = (HEAD_TILE_SIZE, ict.UV_TILES * HEAD_TILE_SIZE)
        baked["head"][:, :, : ict_tiles[1]] = resized(head["texture"], ict_tiles)


light = torch.nn.Parameter(uniform_light())
wrinkle_maps = (
    torch.from_numpy(head["wrinkles"]).float().to(device)
    if head is not None
    else torch.zeros(E_HEAD, 8, 8, 3, device=device)
)


def render(vertices, camera, body_texture, head_texture, light, weights=None):
    clip = to_clip(vertices, camera)
    rast, rast_db = dr.rasterize(context, clip, tri, (camera.height, camera.width))
    covered = (rast[..., 3] > 0).float()[..., None]
    face = (rast[..., 3].long() - 1).clamp_min(0)
    uv, uv_da = dr.interpolate(corner_uvs, rast, corner_tri, rast_db=rast_db, diff_attrs="all")
    on_head = (face_textures[face] == HEAD_TEXTURE).float()[..., None]
    # The head's atlas is 6 tiles wide: past 8 halvings its short side is down to a texel.
    mipmapped = {"filter_mode": "linear-mipmap-linear", "max_mip_level": 8}
    from_body = dr.texture(body_texture, uv, uv_da, **mipmapped)
    from_head = dr.texture(head_texture, uv, uv_da, **mipmapped)
    colour = torch.lerp(from_body, from_head, on_head)
    if weights is not None:
        on_face = on_head * (uv[..., :1] < 1 / HEAD_TILES).float()
        face_uv = torch.cat([uv[..., :1] * HEAD_TILES, uv[..., 1:]], -1).contiguous()
        wrinkles = torch.einsum("be,ehwc->bhwc", weights, wrinkle_maps).contiguous()
        colour = colour + on_face * dr.texture(wrinkles, face_uv, filter_mode="linear")
    colour = colour.clamp(0, 1) * covered
    normals, _ = dr.interpolate(vertex_normals(vertices).contiguous(), rast, tri)
    shading = (sh_basis(F.normalize(normals, dim=-1)) @ light).clamp_min(0) * covered
    lit = dr.antialias((colour * shading).contiguous(), rast, clip, tri)
    alpha = dr.antialias(covered.contiguous(), rast, clip, tri)[..., 0]
    part = torch.where(covered[..., 0] > 0, face_parts[face], -1)
    return {
        "colour": lit.clamp(0, 1),
        "albedo": colour,
        "covered": covered[..., 0],
        "alpha": alpha,
        "part": part,
    }

## Losses and scores

Per frame: colour (inside the eroded mask, where the avatar is drawn, hands left out) and the
outline. Over the video: the corrections stay small and change smoothly, the face sparse, the
shape changes small. PSNR is scored on the performer only (hands included, so the number is
honest about them), with the outline's IoU.

In [ ]:
def shared_state(detached):
    state = {
        "rest": rest_vertices(),
        "body_texture": baked["body"],
        "head_texture": baked["head"],
        "light": light,
    }
    return {k: v.detach() for k, v in state.items()} if detached else state


def frame_terms(frames, shared):
    weights = expressions()[frames]
    rotations, translations = frame_pose(frames)
    vertices = posed_vertices(shared["rest"], weights, rotations, translations)
    out = render(
        vertices, camera, shared["body_texture"], shared["head_texture"], shared["light"], weights
    )
    region = inner[frames] * out["covered"] * (out["part"] != HANDS).float()
    error = (out["colour"] - images[frames]).abs().mean(-1)
    photo = (error * region).sum() / region.sum().clamp_min(1)
    silhouette = (out["alpha"] - masks[frames]).square().mean()
    return {"photo": photo, "silhouette": silhouette}, out


def video_terms():
    return {
        "pose_prior": rotation_corrections.square().sum((1, 2)).mean()
        + (10 * translation_corrections).square().sum(-1).mean(),
        "temporal": second_difference(rotation_corrections).square().sum((1, 2)).mean()
        + second_difference(10 * translation_corrections).square().sum(-1).mean()
        + 0.01 * expressions().diff(dim=0).square().sum(-1).mean(),
        "sparsity": expressions().sum(-1).mean(),
        "offsets": body_offsets().square().sum(-1).mean(),
    }


@torch.no_grad()
def scores():
    shared = shared_state(detached=True)
    psnr, iou = torch.zeros(FRAMES), torch.zeros(FRAMES)
    for frames, _ in chunks:
        _, out = frame_terms(frames, shared)
        region = inner[frames] * out["covered"]
        mse = ((out["colour"] - images[frames]).square().mean(-1) * region).sum(
            (1, 2)
        ) / region.sum((1, 2)).clamp_min(1)
        psnr[frames] = (-10 * torch.log10(mse)).cpu()
        drawn, mask = out["alpha"] > 0.5, masks[frames] > 0.5
        iou[frames] = ((drawn & mask).sum((1, 2)) / (drawn | mask).sum((1, 2)).clamp_min(1)).cpu()
    return {"psnr": psnr.numpy(), "iou": iou.numpy()}


def summary(result, label):
    for part, chosen in (("trained", ~held_out.numpy()), ("held out", held_out.numpy())):
        psnr, iou = result["psnr"][chosen].mean(), result["iou"][chosen].mean()
        print(f"{label:>8} {part:>8}: PSNR {psnr:6.2f} dB, IoU {iou:.3f}")

## Baking the albedo

Each training frame is drawn with the current poses; every pixel inside the eroded mask is
divided by the light its surface receives and splatted into the texel it shows (bilinearly),
weighted by how squarely the surface faces the camera. Texels with little or no weight take
their colour from coarser levels of the same bake (push-pull), so unseen areas get their
surroundings' colour, not a hole.

In [ ]:
def push_pull(sums, weights):
    # (H, W, 3) weighted sums, (H, W) weights -> (1, H, W, 3) averages, gaps filled from coarser
    levels = [(sums.permute(2, 0, 1)[None], weights[None, None])]
    while max(levels[-1][1].shape[-2:]) > 1:
        s, w = levels[-1]
        shape = (max(s.shape[-2] // 2, 1), max(s.shape[-1] // 2, 1))
        levels.append((F.interpolate(s, shape, mode="area"), F.interpolate(w, shape, mode="area")))
    s, w = levels[-1]
    value = s / w.clamp_min(1e-8) if w.max() > 0 else s
    for s, w in reversed(levels[:-1]):
        coarser = F.interpolate(value, s.shape[-2:], mode="bilinear", align_corners=False)
        trust = w / (w + FILL_WEIGHT)
        value = trust * s / w.clamp_min(1e-8) + (1 - trust) * coarser
    return value.permute(0, 2, 3, 1).clamp(0, 1).contiguous()


@torch.no_grad()
def bake(shared):
    # {BODY_TEXTURE: (1, *BODY_SHAPE, 3), HEAD_TEXTURE: (1, *HEAD_SHAPE, 3)} albedo from the frames
    shapes = {BODY_TEXTURE: BODY_SHAPE, HEAD_TEXTURE: HEAD_SHAPE}
    sums = {t: torch.zeros(h * w, 3, device=device) for t, (h, w) in shapes.items()}
    totals = {t: torch.zeros(h * w, device=device) for t, (h, w) in shapes.items()}
    for frames, held in chunks:
        if held:
            continue
        rotations, translations = frame_pose(frames)
        vertices = posed_vertices(shared["rest"], expressions()[frames], rotations, translations)
        clip = to_clip(vertices, camera)
        rast, _ = dr.rasterize(context, clip, tri, (camera.height, camera.width))
        face = rast[..., 3].long() - 1
        uv, _ = dr.interpolate(corner_uvs, rast, corner_tri)
        normals, _ = dr.interpolate(vertex_normals(vertices).contiguous(), rast, tri)
        normals = F.normalize(normals, dim=-1)
        points, _ = dr.interpolate(vertices.contiguous(), rast, tri)
        facing = (normals * F.normalize(camera.position - points, dim=-1)).sum(-1).clamp_min(0)
        shading = sh_basis(normals) @ shared["light"]
        usable = (face >= 0) & (inner[frames] > 0.5) & (shading.min(dim=-1).values > MIN_SHADING)
        albedo = images[frames] / shading.clamp_min(MIN_SHADING)
        for texture_id, (height_, width_) in shapes.items():
            chosen = usable & (face_textures[face.clamp_min(0)] == texture_id)
            x = uv[..., 0][chosen] * width_ - 0.5
            y = uv[..., 1][chosen] * height_ - 0.5
            x0, y0 = x.floor(), y.floor()
            fx, fy = x - x0, y - y0
            value, weight = albedo[chosen], facing[chosen].square()
            for dx, dy, share in (
                (0, 0, (1 - fx) * (1 - fy)),
                (1, 0, fx * (1 - fy)),
                (0, 1, (1 - fx) * fy),
                (1, 1, fx * fy),
            ):
                column = (x0 + dx).long().clamp(0, width_ - 1)
                row = (y0 + dy).long().clamp(0, height_ - 1)
                index = row * width_ + column
                sums[texture_id].index_add_(0, index, value * (weight * share)[:, None])
                totals[texture_id].index_add_(0, index, weight * share)
    return {
        t: push_pull(sums[t].view(h, w, 3), totals[t].view(h, w)) for t, (h, w) in shapes.items()
    }


def rebake():
    result = bake(shared_state(detached=True))
    baked["body"] = result[BODY_TEXTURE]
    if head is None:  # the whole head atlas comes from this video
        baked["head"] = result[HEAD_TEXTURE]
    elif HEAD_TILES == ATLAS_TILES:  # only the genitals' tile, which notebook 08 does not make
        genitals = slice(ict.UV_TILES * HEAD_TILE_SIZE, None)
        baked["head"][:, :, genitals] = result[HEAD_TEXTURE][:, :, genitals]


rebake()

## Showing frames: whole, and around the head

In [ ]:
@torch.no_grad()
def head_box(vertices, margin=1.3):
    points = camera.project(vertices[HEAD][: ict.SKIN.stop])
    centre = (points.min(0).values + points.max(0).values) / 2
    half = margin * (points.max(0).values - points.min(0).values).max() / 2
    x0, y0 = (centre - half).clamp_min(0).long().tolist()
    x1, y1 = (centre + half).long().tolist()
    return slice(y0, min(y1, camera.height)), slice(x0, min(x1, camera.width))


@torch.no_grad()
def show(picks=(0, 12, 24, 47, 60), title=""):
    shared = shared_state(detached=True)
    fig, axes = plt.subplots(len(picks), 6, figsize=(18, 4.2 * len(picks)))
    for row, i in enumerate(picks):
        frames = torch.tensor([min(i, FRAMES - 1)], device=device)
        _, out = frame_terms(frames, shared)
        rotations, translations = frame_pose(frames)
        vertices = posed_vertices(shared["rest"], expressions()[frames], rotations, translations)[0]
        crop = head_box(vertices)
        image = images[frames[0]].cpu().numpy()
        rendered = out["colour"][0].cpu().numpy()
        albedo = out["albedo"][0].cpu().numpy()
        error = np.abs(rendered - image).mean(-1) * out["covered"][0].cpu().numpy()
        held = " (held out)" if held_out[frames[0].item()] else ""
        for ax, picture, label in zip(
            axes[row],
            (image, rendered, albedo, error, image[crop], rendered[crop]),
            ("frame", "render", "albedo", "|error|", "head", "head render"),
            strict=True,
        ):
            ax.imshow(
                picture,
                cmap="magma" if picture.ndim == 2 else None,
                vmin=0,
                vmax=0.3 if picture.ndim == 2 else None,
            )
            ax.set_title(f"{int(frames[0])}{held}: {label}", fontsize=9)
            ax.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


show(title="the fit as it is: albedo baked under a uniform light")

## Optimisation

In [ ]:
parameters_of = {
    "light": [light],
    "pose": [rotation_corrections, translation_corrections],
    "expressions": [expression_logits],
    "offsets": [offset_values],
}
groups = {
    "appearance": ["light"],
    "pose": ["light", "pose", "expressions"],
    "shape": ["light", "pose", "expressions", "offsets"],
}
history = []


def set_trainable(trainable):
    # parameters a phase does not train get no gradients at all, which saves their backward
    for group, parameters in parameters_of.items():
        for p in parameters:
            p.requires_grad_(group in trainable)


def backward(loss):
    if loss.requires_grad:
        loss.backward()


def iteration_step():
    # one iteration's gradients; returns the logged terms
    # The shared values (the smoothed body) are built once; every chunk
    # renders from detached copies, whose gradients are pushed back through them at the end.
    shared = shared_state(detached=False)
    leaves = {k: v.detach().requires_grad_(v.requires_grad) for k, v in shared.items()}
    logged = {}
    for frames, held in chunks:
        used = {k: v.detach() for k, v in leaves.items()} if held else leaves
        terms, _ = frame_terms(frames, used)
        share = len(frames) / FRAMES
        backward(sum(WEIGHTS[k] * t for k, t in terms.items()) * share)
        for k, t in terms.items():
            logged[k] = logged.get(k, 0.0) + t.item() * share
    pairs = [(shared[k], leaves[k].grad) for k in shared if leaves[k].grad is not None]
    if pairs:
        torch.autograd.backward([value for value, _ in pairs], [grad for _, grad in pairs])
    regular = video_terms()
    backward(sum(WEIGHTS[k] * t for k, t in regular.items()))
    logged.update({k: t.item() for k, t in regular.items()})
    return logged


def run_phase(phase):
    set_trainable(groups[phase])
    optimizer = torch.optim.Adam(
        [{"params": parameters_of[g], "lr": LEARNING_RATES[g]} for g in groups[phase]]
    )
    started = time.time()
    for iteration in range(ITERATIONS[phase]):
        if iteration % REBAKE_EVERY == 0:
            rebake()
        optimizer.zero_grad()
        logged = iteration_step()
        optimizer.step()
        history.append({"phase": phase, **logged})
        if iteration % 50 == 0 or iteration == ITERATIONS[phase] - 1:
            print(
                f"{phase} {iteration:4d} ({time.time() - started:5.0f} s): "
                + ", ".join(f"{k} {v:.2e}" for k, v in logged.items())
            )

### Profiling (optional)

Times one iteration of every phase (gradients only, no step, nothing learned), then lists the
GPU operations of the slowest one. Set `PROFILE = False` to skip it.

In [ ]:
PROFILE = True


def clear_gradients():
    for parameters in parameters_of.values():
        for p in parameters:
            p.grad = None


if PROFILE:
    timings = {}
    for phase in groups:
        set_trainable(groups[phase])
        iteration_step()  # warm-up: kernels compile and caches fill
        torch.cuda.synchronize()
        started = time.time()
        iteration_step()
        torch.cuda.synchronize()
        timings[phase] = time.time() - started
        clear_gradients()
    for phase, seconds in timings.items():
        minutes = seconds * ITERATIONS[phase] / 60
        print(f"{phase:>12}: {seconds:5.2f} s per iteration, {minutes:5.1f} min for the phase")
    slowest = max(timings, key=timings.get)
    set_trainable(groups[slowest])
    activities = [torch.profiler.ProfilerActivity.CPU, torch.profiler.ProfilerActivity.CUDA]
    with torch.profiler.profile(activities=activities) as profiler:
        iteration_step()
        torch.cuda.synchronize()
    clear_gradients()
    print(f"\nthe slowest phase, {slowest}, by GPU time:")
    print(profiler.key_averages().table(sort_by="cuda_time_total", row_limit=20))

### Phase 1: the light, with the albedo baked again as it changes

In [ ]:
run_phase("appearance")
before = scores()
summary(before, "before")
show(title="after phase 1: fitted poses, baked albedo, learned light")

### Phase 2: + per-frame pose corrections and the face

In [ ]:
run_phase("pose")
summary(scores(), "phase 2")
show(title="after phase 2")

### Phase 3: + the body's shape changes

In [ ]:
run_phase("shape")
rebake()
after = scores()
summary(before, "before")
summary(after, "after")
show(title="after phase 3")

## Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 3.5))
for ax, key, label in zip(axes, ("psnr", "iou"), ("PSNR (dB)", "IoU"), strict=True):
    ax.plot(before[key], "--", label="before")
    (line,) = ax.plot(after[key], label="after")
    held = np.flatnonzero(held_out.numpy())
    ax.plot(held, after[key][held], "o", color=line.get_color(), markersize=4)
    ax.set_title(label + " per frame (dots: held out)")
    ax.set_xlabel("frame")
axes[0].legend()
plt.show()

with torch.no_grad():
    turned = (rotation_corrections * posed).norm(dim=-1).max(dim=0).values
order = torch.argsort(-turned)[:15].tolist()
plt.figure(figsize=(12, 3))
plt.bar(range(len(order)), np.degrees(turned[order].cpu().numpy()))
plt.xticks(range(len(order)), [body.bone_labels[i] for i in order], rotation=60, fontsize=8)
plt.ylabel("largest correction (°)")
plt.title("the bones the refinement turned most")
plt.show()

with torch.no_grad():
    for name in ("body", "head"):
        plt.figure(figsize=(16 if name == "head" else 8, 4 if name == "head" else 8))
        plt.imshow(baked[name][0].clamp(0, 1).cpu().numpy())
        plt.title(f"{name} albedo")
        plt.axis("off")
        plt.show()

The refined avatar in its rest pose, turning, under the learned light and under a uniform one
(the albedo); and how far the body's shape moved, in millimetres.

In [ ]:
YAWS = np.linspace(0, 2 * np.pi, 6, endpoint=False)


@torch.no_grad()
def turntable(shared, lit=True, colour_by=None, yaws=YAWS):
    count = len(yaws)
    rotations = torch.zeros(count, body.num_bones, 3, device=device)
    turn = roma.rotvec_to_rotmat(
        torch.from_numpy(yaws).float().to(device)[:, None]
        * torch.tensor([0, 0, 1.0], device=device)
    )
    rotations[:, 0] = roma.rotmat_to_rotvec(turn @ roma.rotvec_to_rotmat(fit_rotations[0, 0]))
    translations = fit_translations[:1].expand(count, 3)
    vertices = posed_vertices(
        shared["rest"], torch.zeros(count, E_HEAD, device=device), rotations, translations
    )
    if colour_by is not None:
        clip = to_clip(vertices, camera)
        rast, _ = dr.rasterize(context, clip, tri, (camera.height, camera.width))
        value, _ = dr.interpolate(colour_by[None, :, None].contiguous(), rast, tri)
        return (value[..., 0] * (rast[..., 3] > 0)).cpu().numpy()
    light_ = shared["light"] if lit else uniform_light()
    return (
        render(vertices, camera, shared["body_texture"], shared["head_texture"], light_)["colour"]
        .cpu()
        .numpy()
    )


final = shared_state(detached=True)
moved = torch.cat(
    [body_offsets().detach().norm(dim=-1), torch.zeros(len(rest) - HEAD_START, device=device)]
)
rows = (
    ("learned light", turntable(final)),
    ("albedo", turntable(final, lit=False)),
    ("shape change (mm)", turntable(final, colour_by=moved)),
)
fig, axes = plt.subplots(3, 6, figsize=(18, 14))
for r, (label, pictures) in enumerate(rows):
    for c in range(6):
        scalar = pictures.ndim == 3
        axes[r, c].imshow(
            pictures[c],
            cmap="viridis" if scalar else None,
            vmin=0,
            vmax=float(moved.max()) if scalar else None,
        )
        axes[r, c].axis("off")
    axes[r, 0].set_title(label, loc="left")
plt.tight_layout()
plt.show()
print(f"body shape change: mean {moved[:HEAD_START].mean():.2f} mm, max {moved.max():.2f} mm")

## Export for the viewer

The refined avatar as `outputs/export/refined.avatar`: the mesh with the refined head and body,
the albedo textures, the lashes and brows as `go-task export` makes them (the eyes and mouth
from the texture unless `STOCK_EYES_AND_MOUTH`), no Gaussians, and, new in format 4, the learned light (`lighting`) and the wrinkle maps
(`wrinkles`). Start the viewer (`go-task viewer`) and drop the file on the page; the joint and
expression sliders pose it.

In [ ]:
with torch.no_grad():
    refined = dataclasses.replace(
        avatar,
        rest_vertices=final["rest"].cpu().numpy(),
        expression_basis=basis.cpu().numpy(),
    )
    textures = TextureRecord(
        body=final["body_texture"][0].clamp(0, 1).cpu().numpy(),
        head=final["head_texture"][0].clamp(0, 1).cpu().numpy(),
        body_coverage=np.ones(BODY_SHAPE, np.float32),
        head_coverage=np.ones(HEAD_SHAPE, np.float32),
    )

static = Preprocessed.load(preprocessed_path(config, "head_static"))
static_frames = np.stack(list(read_frames(config.capture.head_static)))[static.frame_indices]
eyes = eye_metadata(
    refined,
    measure_eyes(static_frames, static.face_landmarks),
    ict_landmark_vertices(config.body_models.ict_facekit),
)
hair = grow_face_hair(
    refined,
    textures,
    config.body_models.ict_facekit,
    identity.cpu().numpy(),
    np.random.default_rng(0),
)
hair_entry = face_hair_metadata(measure_hair_colours(static_frames, static.face_landmarks))
no_gaussians = GaussianRecord(
    faces=np.zeros(0, np.int64),
    barycentric=np.zeros((0, 3), np.float32),
    offsets=np.zeros((0, 3), np.float32),
    rotations=np.zeros((0, 4), np.float32),
    log_scales=np.zeros((0, 3), np.float32),
    opacity_logits=np.zeros(0, np.float32),
    sh=np.zeros((0, 1, 3), np.float32),
)
file = avatar_arrays(
    refined, no_gaussians, textures, config.training.occlusion_margin, eyes, hair, hair_entry
)
file.metadata["format_version"] = 4
if not STOCK_EYES_AND_MOUTH:  # the viewer draws what the file does not describe from textures
    del file.metadata["eyes"], file.metadata["mouth"]
file.metadata["lighting"] = {"sh": light.detach().cpu().numpy().tolist()}
maps = wrinkle_maps.cpu().numpy()
wrinkle_scale = float(max(np.abs(maps).max(), 1e-6))
file.metadata["wrinkles"] = {
    "expressions": [str(name) for name in avatar.expression_names[:E_HEAD]],
    "scale": wrinkle_scale,
    "region": [0.0, 0.0, 1 / HEAD_TILES, 1.0],
    "size": int(maps.shape[1]),
}
file.arrays["wrinkle_textures"] = (
    np.round(128 + 127 * maps / wrinkle_scale).clip(0, 255).astype(np.uint8)
)

path = config.outputs / "export" / "refined.avatar"
write_avatar(path, file)
np.savez_compressed(
    out_dir / "body.npz",
    rotation_corrections=rotation_corrections.detach().cpu().numpy(),
    translation_corrections=translation_corrections.detach().cpu().numpy(),
    expressions=expressions().detach().cpu().numpy(),
    body_offsets=body_offsets().detach().cpu().numpy() * MILLIMETRE,
    light=light.detach().cpu().numpy(),
    before_psnr=before["psnr"],
    after_psnr=after["psnr"],
    before_iou=before["iou"],
    after_iou=after["iou"],
    held_out=held_out.numpy(),
)
print(f"wrote {path} ({path.stat().st_size / 1e6:.0f} MB)")